<a href="https://colab.research.google.com/github/Ar1n382/gmt-projects-/blob/main/2230674038_GMT312_Assigment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import numpy as np
import pandas as pd

def xyz2plh(cart): #creating the function


    X = cart[0]
    Y = cart[1]
    Z = cart[2]  #define the coordinates


    #define the Wgs 84 ellipsoid parameters

    a = 6378137.0 #m semi major axis
    FF = 298.257223563 # Flattning factor
    b = a * (1 - FF) #m semi minor axis
    esqrd = 2/FF - (1/FF) **2 #Eccentricity

    #calculatin the long
    Long_Rad = math.atan2(Y, X)
    #calculating the distances from the z axis
    p = math.sqrt(X**2 + Y**2)

    #ittartaive calculation

    Lat0 = math.atan(Z / (p * (1 - esqrd))) #initail approx lat value

    threshold = 1e-12            # Step 2 threshold variables
    diff = 1.0

    while diff >= threshold:
      Nk = a / math.sqrt(1 - esqrd * math.sin(Lat0) **2)  #is radius of curvature in the prime vertical:
      hk = (p / math.cos(Lat0)) - Nk #heigiht
      UP_Lat = math.atan(Z / (p * (1 - esqrd * (Nk / (Nk + hk))))) #updated lat from the ittiration
      diff = abs(UP_Lat - Lat0) #calculating the difference
      Lat0 = UP_Lat

      #COnvert Lat and Long to  radians to deg
      Lat = math.degrees(Lat0)
      Long = math.degrees(Long_Rad)

      #Return the output vector
      ellp = [Lat, Long, hk]

      return ellp

In [ ]:
xyz2plh([100, 200, 500])

[-1.6690920470851291, 63.43494882292201, -6395491.66940705]

In [ ]:
def local(rec, sat):
    #   rec: A vector (3x1) containing geocentric Cartesian coordinates of the
    #        local origin (topocenter) in meters.
    #   sat: A vector (3x1) containing geocentric Cartesian coordinates of the
    #        target point in meters.
    # Outputs:
    #   az: azimuth angle of the target in Degree [0, 360].
    #   zen: zenith angle of the target in Degree [-90,90].
    #   slantd: slant (radial) range from the topocenter to the target in Meter.

    ellp = xyz2plh(rec) #call the prev fuction to get the alşlipsoodial coordinates of the topocenter

    #conver lat long to radians for trginomtric functions
    Lat_Rad = math.radians(ellp[0])
    Long_Rad = math.radians(ellp[1])

    #compıte the cartesian coordiante system diffrances from target to local origin
    dx = sat[0] - rec[0]
    dy = sat[1] - rec[1]
    dz = sat[2] - rec[2]

    #convert the cartesian coordiante sytems to local ellipsodial system (east north up)
    sin_Lat = math.sin(Lat_Rad)
    cos_Lat = math.cos(Lat_Rad)
    sin_Long = math.sin(Long_Rad)
    cos_Long = math.cos(Long_Rad)

    E = -sin_Long * dx + cos_Long * dy
    N = -sin_Lat * cos_Long * dx - sin_Lat * sin_Long * dy + cos_Lat * dz
    U =  cos_Lat * cos_Long * dx + cos_Lat * sin_Long * dy + sin_Lat * dz

    #calculate the salant distance
    slantd = math.sqrt(E**2 + N**2+ U**2)
    #calculate the azimuths
    az = math.degrees(math.atan2(E, N))
    #calculate the zenith
    zen = 90.0 - math.degrees(math.acos(U / slantd))
    #bcs the assigment requries the zenith to be Degree [-90,90] we constrait the zenith angle to fit
    if zen > 90:
        zen = zen - 180

    return [az, zen, slantd]

In [ ]:
local((500,600,300),(5000,2000,109283))

[-1.345676981096691, 1.6385574530907263, 109084.84903505161]